# Evaluate Climate Models

### Parameter Settings: Change Timestep here

In [1]:
# %% [Setup — Climate Model Evaluation]

import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"

import sys
from pathlib import Path

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg

FIG_SUBDIR = "climate_model_evaluation"
FIG_DIRS = [
    cfg.FIGURES_DIR           / FIG_SUBDIR,
    cfg.FIGURES_DIR_SECONDARY / FIG_SUBDIR,
]
for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)

In [2]:
# %% [Load per-catchment data for evaluation figures]
#
# Loads annual maxima and daily values per catchment, per model, for the
# evaluation period. Results are used by:
#   - make_distribution_figure  (distribution analysis plots)
#   - make_qq_figure            (Q-Q mapping plots)
#   - build_percentile_mapping_table / build_distribution_summary_table

from catchment_tools import (
    load_annual_maxima_per_catchment,
    load_daily_values_per_catchment,
    build_percentile_mapping_table,
    build_distribution_summary_table,
)
from plot_style import make_distribution_figure, make_qq_figure

# ── Evaluation period ─────────────────────────────────────────────────────────
# Must be contained within both the SMILE and reanalysis cached ranges.
# Adjust if your cached data uses a different overlap period.
EVAL_START_YEAR = 1985
EVAL_END_YEAR   = 2024
EVAL_PERIOD_TAG = f"{EVAL_START_YEAR}-{EVAL_END_YEAR}"

REANALYSIS_KEYS     = ["senorge", "era5_0.25", "era5_0.5"]
PERCENTILES_TO_COMPARE = (2.5, 5, 16, 50, 84, 95, 97.5)

# ── Load annual maxima per catchment ─────────────────────────────────────────
print("Loading 1-day annual maxima per catchment ...")
am_1day_per_catchment = load_annual_maxima_per_catchment(
    window_days = 1,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

print("Loading 2-day annual maxima per catchment ...")
am_2day_per_catchment = load_annual_maxima_per_catchment(
    window_days = 2,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

# ── Load daily values per catchment ──────────────────────────────────────────
print("Loading 1-day daily values per catchment ...")
daily_1day_per_catchment = load_daily_values_per_catchment(
    window_days = 1,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

print("Loading 2-day daily values per catchment ...")
daily_2day_per_catchment = load_daily_values_per_catchment(
    window_days = 2,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

print(f"\n✓ Done. Eval period: {EVAL_PERIOD_TAG}")
print(f"  Catchments : {list(cfg.CATCHMENTS.keys())}")
for slug in cfg.CATCHMENTS:
    models_1day = list(am_1day_per_catchment[slug].keys())
    print(f"  {slug}: {models_1day}")

Loading 1-day annual maxima per catchment ...
Loading 2-day annual maxima per catchment ...
Loading 1-day daily values per catchment ...


/nird/home/lbal/internship_storm_hans/helper/catchment_tools.py:866: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  with xr.open_dataset(str(member_cache), use_cftime=True) as ds_m:
/nird/home/lbal/internship_storm_hans/helper/catchment_tools.py:866: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  with xr.open_dataset(str(member_cache), use_cftime=True) as ds_m:
/nird/home/lbal/internship_storm_hans/helper/catchment_tools.py:866: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. 

Loading 2-day daily values per catchment ...


/nird/home/lbal/internship_storm_hans/helper/catchment_tools.py:866: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  with xr.open_dataset(str(member_cache), use_cftime=True) as ds_m:
/nird/home/lbal/internship_storm_hans/helper/catchment_tools.py:866: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  with xr.open_dataset(str(member_cache), use_cftime=True) as ds_m:
/nird/home/lbal/internship_storm_hans/helper/catchment_tools.py:866: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. 


✓ Done. Eval period: 1985-2024
  Catchments : ['nevina_bergheim', 'nevina_honnefoss', 'nevina_losna', 'regine_drammen', 'regine_glomma']
  nevina_bergheim: ['era5_0.5', 'era5_0.25', 'senorge', 'cesm2_le', 'gfdl_spear_med_le']
  nevina_honnefoss: ['era5_0.5', 'era5_0.25', 'senorge', 'cesm2_le', 'gfdl_spear_med_le']
  nevina_losna: ['era5_0.5', 'era5_0.25', 'senorge', 'cesm2_le', 'gfdl_spear_med_le']
  regine_drammen: ['era5_0.5', 'era5_0.25', 'senorge', 'cesm2_le', 'gfdl_spear_med_le']
  regine_glomma: ['era5_0.5', 'era5_0.25', 'senorge', 'cesm2_le', 'gfdl_spear_med_le']


/nird/home/lbal/internship_storm_hans/helper/catchment_tools.py:866: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  with xr.open_dataset(str(member_cache), use_cftime=True) as ds_m:


### Create Distribution plots

In [3]:
# %% [Distribution Analysis — per catchment, per window, annual_max + daily]
#
# Produces one PDF per (catchment × window_days × data_type) combination.
#
# Filename pattern:
#   {data_type}_distribution_{window_days}day_{slug}_{start}-{end}.pdf
#
# Saved to: FIG_DIRS (climate_model_evaluation/, NO subfolder)

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc, daily_pc in [
        (1, am_1day_per_catchment, daily_1day_per_catchment),
        (2, am_2day_per_catchment, daily_2day_per_catchment),
    ]:
        for data_type, pc in [
            ("annual_max", am_pc),
            ("daily",      daily_pc),
        ]:
            data_for_catchment = pc[slug]

            if not data_for_catchment:
                print(f"  [skip] No data for {slug} / {window_days}day / {data_type}")
                continue

            fname = (
                f"{data_type}_distribution_{window_days}day_"
                f"{slug}_{EVAL_PERIOD_TAG}.pdf"
            )
            out_paths = [d / fname for d in FIG_DIRS]

            print(f"  Saving: {fname}")
            make_distribution_figure(
                annual_maxima   = data_for_catchment,
                window_days     = window_days,
                out_paths       = out_paths,
                data_type       = data_type,
                catchment_title = catchment_title,
            )

print(f"\n[distribution] ✓ Done. Figures saved to:")
for d in FIG_DIRS:
    print(f"  {d}")

  Saving: annual_max_distribution_1day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/annual_max_distribution_1day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/annual_max_distribution_1day_nevina_bergheim_1985-2024.pdf
  Saving: daily_distribution_1day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/daily_distribution_1day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/daily_distribution_1day_nevina_bergheim_1985-2024.pdf
  Saving: annual_max_distribution_2day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/annual_max_distribution_2day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluati

### Create Q-Q mapping Plots

In [4]:
# Per-catchment QQ plots: CESM2-LE and GFDL-SPEAR vs. Reanalysis
# Runs for both annual_max and daily data types.

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc, daily_pc in [
        (1, am_1day_per_catchment, daily_1day_per_catchment),
        (2, am_2day_per_catchment, daily_2day_per_catchment),]:
        for data_type, pc in [
            ("annual_max", am_pc),
            ("daily",      daily_pc),]:
            for climate_key in ("cesm2_le", "gfdl_spear_med_le"):
                reanalysis = {k: pc[slug][k] for k in REANALYSIS_KEYS if k in pc[slug]}
                out = [
                    d / f"qq-plot_{data_type}_{climate_key}_{window_days}day_{slug}_{EVAL_PERIOD_TAG}.pdf"
                    for d in FIG_DIRS]
                make_qq_figure(
                    climate_key,
                    pc[slug][climate_key],
                    reanalysis,
                    window_days=window_days,
                    out_paths=out,
                    data_type=data_type,
                    catchment_title=catchment_title,)

    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/qq-plot_annual_max_cesm2_le_1day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/qq-plot_annual_max_cesm2_le_1day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/qq-plot_annual_max_gfdl_spear_med_le_1day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/qq-plot_annual_max_gfdl_spear_med_le_1day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/qq-plot_daily_cesm2_le_1day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/qq-plot_daily_cesm2_le_1day_nevina_bergheim_1985-2024.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/qq-plot_daily_gfd

### Create Statistical Analysis for QQ-Plot

In [5]:
# Per-catchment percentile mapping tables

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc in [
        (1, am_1day_per_catchment),
        (2, am_2day_per_catchment),]:
        refs = {k: am_pc[slug][k] for k in REANALYSIS_KEYS if k in am_pc[slug]}

        for climate_key in ("cesm2_le", "gfdl_spear_med_le"):
            df_pct = build_percentile_mapping_table(
                climate_key,
                am_pc[slug][climate_key],
                refs,
                percentiles=PERCENTILES_TO_COMPARE,
            ).round(1)

            print(f"\nPercentile mapping: {climate_key} | {window_days}-day | {slug} | {EVAL_PERIOD_TAG}")
            display(df_pct)

            for d in FIG_DIRS:
                out = d / f"percentile_mapping_{climate_key}_{window_days}day_{slug}_{EVAL_PERIOD_TAG}.csv"
                df_pct.to_csv(out, index=False)
                print(f"Saved -> {out}")


Percentile mapping: cesm2_le | 1-day | nevina_bergheim | 1985-2024


,climate_model,target_percentile_%,climate_value_mm,senorge_percentile_%,era5_0.25_percentile_%,era5_0.5_percentile_%
0,cesm2_le,2.5,20.9,0.0,5.0,5.0
1,cesm2_le,5.0,21.9,2.5,5.0,7.5
2,cesm2_le,16.0,24.4,12.5,17.5,17.5
3,cesm2_le,50.0,29.2,42.5,40.0,42.5
4,cesm2_le,84.0,35.9,67.5,77.5,80.0
5,cesm2_le,95.0,42.0,90.0,95.0,97.5
6,cesm2_le,97.5,46.0,92.5,100.0,100.0


Saved -> /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/percentile_mapping_cesm2_le_1day_nevina_bergheim_1985-2024.csv
Saved -> /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/percentile_mapping_cesm2_le_1day_nevina_bergheim_1985-2024.csv

Percentile mapping: gfdl_spear_med_le | 1-day | nevina_bergheim | 1985-2024


,climate_model,target_percentile_%,climate_value_mm,senorge_percentile_%,era5_0.25_percentile_%,era5_0.5_percentile_%
0,gfdl_spear_med_le,2.5,19.7,0.0,0.0,2.5
1,gfdl_spear_med_le,5.0,20.9,0.0,5.0,5.0
2,gfdl_spear_med_le,16.0,24.1,10.0,12.5,15.0
3,gfdl_spear_med_le,50.0,30.7,45.0,47.5,47.5
4,gfdl_spear_med_le,84.0,40.4,87.5,92.5,92.5
5,gfdl_spear_med_le,95.0,48.5,97.5,100.0,100.0
6,gfdl_spear_med_le,97.5,51.8,97.5,100.0,100.0


Saved -> /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/percentile_mapping_gfdl_spear_med_le_1day_nevina_bergheim_1985-2024.csv
Saved -> /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/percentile_mapping_gfdl_spear_med_le_1day_nevina_bergheim_1985-2024.csv

Percentile mapping: cesm2_le | 2-day | nevina_bergheim | 1985-2024


...

Saved -> /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/percentile_mapping_gfdl_spear_med_le_1day_regine_glomma_1985-2024.csv
Saved -> /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/percentile_mapping_gfdl_spear_med_le_1day_regine_glomma_1985-2024.csv

Percentile mapping: cesm2_le | 2-day | regine_glomma | 1985-2024


,climate_model,target_percentile_%,climate_value_mm,senorge_percentile_%,era5_0.25_percentile_%,era5_0.5_percentile_%
0,cesm2_le,2.5,22.0,0.0,0.0,0.0
1,cesm2_le,5.0,23.2,2.5,0.0,0.0
2,cesm2_le,16.0,26.0,2.5,0.0,0.0
3,cesm2_le,50.0,32.0,32.5,32.5,32.5
4,cesm2_le,84.0,41.0,72.5,70.0,70.0
5,cesm2_le,95.0,49.5,87.5,90.0,90.0
6,cesm2_le,97.5,53.6,92.5,95.0,92.5


Saved -> /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/percentile_mapping_cesm2_le_2day_regine_glomma_1985-2024.csv
Saved -> /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/percentile_mapping_cesm2_le_2day_regine_glomma_1985-2024.csv

Percentile mapping: gfdl_spear_med_le | 2-day | regine_glomma | 1985-2024


,climate_model,target_percentile_%,climate_value_mm,senorge_percentile_%,era5_0.25_percentile_%,era5_0.5_percentile_%
0,gfdl_spear_med_le,2.5,23.0,2.5,0.0,0.0
1,gfdl_spear_med_le,5.0,24.8,2.5,0.0,0.0
2,gfdl_spear_med_le,16.0,29.0,12.5,17.5,17.5
3,gfdl_spear_med_le,50.0,35.2,52.5,47.5,47.5
4,gfdl_spear_med_le,84.0,44.6,77.5,85.0,85.0
5,gfdl_spear_med_le,95.0,54.0,92.5,95.0,95.0
6,gfdl_spear_med_le,97.5,58.7,92.5,97.5,97.5


Saved -> /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/percentile_mapping_gfdl_spear_med_le_2day_regine_glomma_1985-2024.csv
Saved -> /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/percentile_mapping_gfdl_spear_med_le_2day_regine_glomma_1985-2024.csv


In [6]:
# Per-catchment distribution summary tables

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc in [
        (1, am_1day_per_catchment),
        (2, am_2day_per_catchment),]:
        summary = build_distribution_summary_table(am_pc[slug]).round(2)

        print(f"\n{window_days}-day summary | {slug} | {EVAL_PERIOD_TAG}")
        display(summary)

        for d in FIG_DIRS:
            out = d / f"distribution_summary_{window_days}day_{slug}_{EVAL_PERIOD_TAG}.csv"
            summary.to_csv(out, index=False)
            print(f"Saved -> {out}")


1-day summary | nevina_bergheim | 1985-2024


,model,label,n,mean_mm,std_mm,min_mm,q02_5_mm,q05_mm,q16_mm,q25_mm,median_mm,q75_mm,q84_mm,q95_mm,q97_5_mm,max_mm,iqr_mm
0,senorge,SeNorge / 1 km,40,32.55,8.00,21.81,22.00,23.92,24.98,26.06,31.96,36.67,39.00,47.25,47.89,59.28,10.61
1,era5_0.25,ERA5 / 0.25°,40,31.22,6.03,20.17,20.53,22.21,24.70,27.50,31.57,34.85,37.13,41.45,42.14,43.78,7.35
2,era5_0.5,ERA5 / 0.5°,40,30.76,5.96,19.50,19.95,21.71,24.54,27.08,31.16,34.57,36.48,41.32,41.60,43.59,7.48
3,gfdl_spear_med_le,GFDL-SPEAR / 0.5° x 0.625°,1200,32.19,8.58,14.45,19.68,20.86,24.11,26.04,30.67,37.01,40.35,48.45,51.76,86.61,10.97
4,cesm2_le,CESM2-LE / 0.94° x 1.25°,4000,30.23,6.46,16.79,20.87,21.88,24.41,25.78,29.19,33.40,35.90,42.00,45.99,74.42,7.62


Saved -> /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/distribution_summary_1day_nevina_bergheim_1985-2024.csv
Saved -> /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/distribution_summary_1day_nevina_bergheim_1985-2024.csv

2-day summary | nevina_bergheim | 1985-2024


,model,label,n,mean_mm,std_mm,min_mm,q02_5_mm,q05_mm,q16_mm,q25_mm,median_mm,q75_mm,q84_mm,q95_mm,q97_5_mm,max_mm,iqr_mm
0,senorge,SeNorge / 1 km,40,45.80,13.65,31.52,31.60,31.81,36.00,39.35,42.97,49.61,55.59,60.42,71.03,110.48,10.26
1,era5_0.25,ERA5 / 0.25°,40,45.05,9.93,30.41,32.58,34.40,37.12,38.07,42.55,48.80,52.90,66.46,69.79,71.52,10.73
2,era5_0.5,ERA5 / 0.5°,40,44.23,9.73,29.32,33.13,33.75,35.44,37.03,41.88,47.89,52.58,65.80,68.70,69.40,10.87
3,gfdl_spear_med_le,GFDL-SPEAR / 0.5° x 0.625°,1200,44.63,11.49,21.99,27.87,29.92,34.07,36.51,42.61,51.08,55.50,65.76,69.98,137.60,14.58
4,cesm2_le,CESM2-LE / 0.94° x 1.25°,4000,44.17,9.07,26.05,31.35,32.80,36.24,38.04,42.65,48.41,51.94,60.59,65.98,114.35,10.37


Saved -> /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/distribution_summary_2day_nevina_bergheim_1985-2024.csv
Saved -> /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/distribution_summary_2day_nevina_bergheim_1985-2024.csv

1-day summary | nevina_honnefoss | 1985-2024


...

Saved -> /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/distribution_summary_2day_regine_drammen_1985-2024.csv
Saved -> /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/distribution_summary_2day_regine_drammen_1985-2024.csv

1-day summary | regine_glomma | 1985-2024


,model,label,n,mean_mm,std_mm,min_mm,q02_5_mm,q05_mm,q16_mm,q25_mm,median_mm,q75_mm,q84_mm,q95_mm,q97_5_mm,max_mm,iqr_mm
0,senorge,SeNorge / 1 km,40,26.02,6.94,15.29,17.74,18.56,19.83,21.38,24.55,28.89,33.11,38.45,40.83,47.87,7.51
1,era5_0.25,ERA5 / 0.25°,40,25.69,6.83,16.76,18.04,18.12,19.30,20.51,23.65,28.50,33.51,38.93,41.12,41.18,7.98
2,era5_0.5,ERA5 / 0.5°,40,25.81,6.89,16.78,18.16,18.26,19.53,20.63,23.71,28.66,33.83,39.08,41.48,41.73,8.02
3,gfdl_spear_med_le,GFDL-SPEAR / 0.5° x 0.625°,1200,25.65,6.39,12.86,16.01,17.02,19.82,21.28,24.67,28.80,31.33,37.86,40.48,60.09,7.52
4,cesm2_le,CESM2-LE / 0.94° x 1.25°,4000,23.38,5.81,11.83,14.97,15.90,18.11,19.18,22.37,26.36,28.66,34.27,37.38,58.92,7.18


Saved -> /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/distribution_summary_1day_regine_glomma_1985-2024.csv
Saved -> /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/distribution_summary_1day_regine_glomma_1985-2024.csv

2-day summary | regine_glomma | 1985-2024


,model,label,n,mean_mm,std_mm,min_mm,q02_5_mm,q05_mm,q16_mm,q25_mm,median_mm,q75_mm,q84_mm,q95_mm,q97_5_mm,max_mm,iqr_mm
0,senorge,SeNorge / 1 km,40,37.93,10.46,22.97,26.77,27.92,29.37,30.25,35.07,41.81,46.33,58.79,60.57,74.79,11.56
1,era5_0.25,ERA5 / 0.25°,40,37.39,9.40,27.06,27.37,27.48,28.69,30.17,35.67,42.56,43.98,53.48,58.63,72.09,12.39
2,era5_0.5,ERA5 / 0.5°,40,37.59,9.52,27.38,27.43,27.57,28.86,30.30,35.74,42.82,44.21,53.95,58.98,73.00,12.52
3,gfdl_spear_med_le,GFDL-SPEAR / 0.5° x 0.625°,1200,36.70,8.89,18.33,22.98,24.77,28.96,30.56,35.22,41.07,44.59,53.99,58.69,83.36,10.51
4,cesm2_le,CESM2-LE / 0.94° x 1.25°,4000,33.59,8.31,17.29,22.01,23.17,26.03,27.71,32.04,37.70,41.05,49.50,53.63,84.67,9.98


Saved -> /nird/datalake/NS9873K/lbal/figures/climate_model_evaluation/distribution_summary_2day_regine_glomma_1985-2024.csv
Saved -> /nird/home/lbal/internship_storm_hans/figures/climate_model_evaluation/distribution_summary_2day_regine_glomma_1985-2024.csv
